#Lab-1

Task 1

In [ ]:
!pip install diffusers transformers accelerate torch

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
import os

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

In [ ]:
output_dir = "generated_images"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
prompts = [
    "A futuristic city at night",
    "A cat chasing a mouse",
    "A sunset over snowy mountains",
    "A cyberpunk street scene",
    "A lava dragon attacking a castle"
]

In [ ]:
for i, prompt in enumerate(prompts):
    image = pipe(prompt).images[0]
    image.save(f"{output_dir}/image_{i+1}.png")
    display(image)

Task 2

In [ ]:
!pip install diffusers transformers accelerate torch safetensors pillow


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")


In [ ]:
labels = [
    "Normal anatomy – healthy lungs, age & gender variations",
    "Infectious patterns – bacterial/viral pneumonia, COVID-like opacities",
    "Lung opacities – focal, diffuse, ground-glass, consolidations",
    "Pleural conditions – pleural effusion, pneumothorax",
    "Structural lesions – nodules, masses, fibrosis",
    "Cardiac findings – cardiomegaly, vascular congestion",
    "Medical devices – tubes, catheters, pacemakers",
    "Imaging artifacts – noise, motion blur, exposure issues",
    "View & positioning – PA/AP views, rotation, supine/erect",
    "Domain shift – scanner, hospital, and resolution variations"
]


In [ ]:
import os
import re

def clean_label(label):
    return re.sub(r'[^a-zA-Z0-9_]', '_', label.split("–")[0].strip())

base_dir = "/content/xray_dataset"
os.makedirs(base_dir, exist_ok=True)

clean_labels = {}

for label in labels:
    folder = clean_label(label)
    clean_labels[label] = folder
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)


In [ ]:
variation_descriptions = [
    "very mild and early-stage findings",
    "mild but clearly visible clinical findings",
    "moderate severity with noticeable abnormalities",
    "severe and advanced pathological findings",
    "critical condition with extensive abnormalities"
]


In [ ]:
def build_prompt(label, i):
    return f"""
    A high-resolution clinical chest X-ray image showing {label}.
    The case represents {variation_descriptions[i-1]}.
    The image should be medically accurate, realistic, and suitable for diagnostic use.
    Hospital imaging style, sharp anatomical details, proper contrast,
    no artistic effects, no cartoons, professional medical scan.
    """


In [ ]:
from PIL import Image

for label in labels:
    folder = clean_labels[label]

    for i in range(1, 6):
        prompt = build_prompt(label, i)

        image = pipe(
            prompt,
            num_inference_steps=50,
            guidance_scale=8.5
        ).images[0]

        filename = f"{folder}_{i}.png"
        path = os.path.join(base_dir, folder, filename)
        image.save(path)

        print(f"Saved: {path}")


In [ ]:
# Install CLIP properly
!pip install ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git

import torch
import clip
import os
from PIL import Image

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP model
model, preprocess = clip.load("ViT-B/32", device=device)

# Your medical labels
labels = [
    "Normal anatomy – healthy lungs, age & gender variations",
    "Infectious patterns – bacterial/viral pneumonia, COVID-like opacities",
    "Lung opacities – focal, diffuse, ground-glass, consolidations",
    "Pleural conditions – pleural effusion, pneumothorax",
    "Structural lesions – nodules, masses, fibrosis",
    "Cardiac findings – cardiomegaly, vascular congestion",
    "Medical devices – tubes, catheters, pacemakers",
    "Imaging artifacts – noise, motion blur, exposure issues",
    "View & positioning – PA/AP views, rotation, supine/erect",
    "Domain shift – scanner, hospital, and resolution variations"
]

# Encode text labels
text_tokens = clip.tokenize(labels).to(device)
text_features = model.encode_text(text_tokens)

# Dataset path
dataset_path = "/content/xray_dataset"

# Classify all images
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    print(f"\n📁 Folder: {folder}")

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)

        image = preprocess(Image.open(img_path).convert("RGB")).unsqueeze(0).to(device)

        with torch.no_grad():
            image_features = model.encode_image(image)
            similarity = (image_features @ text_features.T).softmax(dim=-1)
            best_match = similarity.argmax().item()

        print(f"{img_name} → {labels[best_match]}")
